# 03.5.0 FastF1 Data Format Analysis

**Purpose**: Investigate data types and formats in FastF1 and master data to identify conversion needs before reproduction.

**Scope**: 
- Inspect FastF1 data formats (Time, milliseconds, etc.)
- Inspect master data formats
- Identify conversion requirements
- Test conversion functions
- Provide standardized format recommendations

**Output**: Conversion requirements and standardized format specifications


## Setup: Imports and Paths


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Get project root
PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

# Paths
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
KAGGLE_DIR = RAW_DATA_DIR / 'kaggle'
FASTF1_DIR = RAW_DATA_DIR / 'fastf1_2018plus'

print("Project Structure:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  FASTF1_DIR: {FASTF1_DIR}")
print(f"  PROCESSED_DATA_DIR: {PROCESSED_DATA_DIR}")
print()

assert FASTF1_DIR.exists(), f"FastF1 data directory not found: {FASTF1_DIR}"
assert PROCESSED_DATA_DIR.exists(), f"Processed data directory not found: {PROCESSED_DATA_DIR}"
print("✓ All directories exist")


Project Structure:
  PROJECT_ROOT: C:\Users\erikv\Downloads\F1
  FASTF1_DIR: C:\Users\erikv\Downloads\F1\data\raw\fastf1_2018plus
  PROCESSED_DATA_DIR: C:\Users\erikv\Downloads\F1\data\processed

✓ All directories exist


## 1. FastF1 Data Format Inspection

Inspect FastF1 data types and formats, especially for time-related columns.


In [2]:
# Load FastF1 2024 data for inspection
year = 2024
results_file = FASTF1_DIR / f'ALL_RESULTS_{year}.csv'

print(f"Loading FastF1 RESULTS for {year}...")
fastf1_results = pd.read_csv(results_file, low_memory=False)
print(f"  ✓ Loaded {len(fastf1_results):,} rows")
print(f"  Sessions: {fastf1_results['Session'].value_counts().to_dict()}")
print()

# Filter to Race session for analysis
race_results = fastf1_results[fastf1_results['Session'] == 'R'].copy()
print(f"Race session rows: {len(race_results):,}")
print(f"\nRace session columns:")
for i, col in enumerate(race_results.columns, 1):
    print(f"  {i:2d}. {col}")


Loading FastF1 RESULTS for 2024...
  ✓ Loaded 2,277 rows
  Sessions: {'FP1': 480, 'R': 479, 'Q': 479, 'FP2': 360, 'FP3': 359, 'Sprint': 120}

Race session rows: 479

Race session columns:
   1. DriverNumber
   2. BroadcastName
   3. Abbreviation
   4. DriverId
   5. TeamName
   6. TeamColor
   7. TeamId
   8. FirstName
   9. LastName
  10. FullName
  11. HeadshotUrl
  12. CountryCode
  13. Position
  14. ClassifiedPosition
  15. GridPosition
  16. Q1
  17. Q2
  18. Q3
  19. Time
  20. Status
  21. Points
  22. Laps
  23. Year
  24. Event
  25. Session


### 1.1 Time Column Format Analysis


In [3]:
print("="*80)
print("FASTF1 TIME COLUMN FORMAT ANALYSIS")
print("="*80)
print()

if 'Time' in race_results.columns:
    time_col = race_results['Time']
    
    print(f"Data type: {time_col.dtype}")
    print(f"Total values: {len(time_col)}")
    print(f"Non-null values: {time_col.notna().sum()}")
    print(f"Null values: {time_col.isna().sum()}")
    print()
    
    # Sample non-null values
    non_null_times = time_col[time_col.notna()]
    print(f"Sample Time values (first 20 non-null):")
    for i, val in enumerate(non_null_times.head(20), 1):
        print(f"  {i:2d}. {repr(val)} (type: {type(val).__name__})")
    print()
    
    # Analyze formats
    print("Format Analysis:")
    
    # Check for gap times (starting with '+')
    time_str = time_col.astype(str)
    gap_times = time_str[time_str.str.strip().str.startswith('+')]
    print(f"  Gap times (starting with '+'): {len(gap_times)} ({len(gap_times)/len(non_null_times)*100:.1f}% if non-null > 0)")
    if len(gap_times) > 0:
        print(f"  Sample gap times:")
        for val in gap_times.head(10):
            print(f"    {repr(val)}")
    print()
    
    # Check for timedelta format ("0 days 00:39:09.686000")
    timedelta_format = time_str[time_str.str.contains('days', na=False)]
    print(f"  Timedelta format (contains 'days'): {len(timedelta_format)} ({len(timedelta_format)/len(non_null_times)*100:.1f}% if non-null > 0)")
    if len(timedelta_format) > 0:
        print(f"  Sample timedelta times:")
        for val in timedelta_format.head(10):
            print(f"    {repr(val)}")
    print()
    
    # Check for absolute time format (HH:MM:SS.mmm or MM:SS.mmm)
    absolute_pattern = r'^\d+:\d{2}:\d{2}\.\d+|^\d+:\d{2}\.\d+'
    absolute_times = time_str[time_str.str.match(absolute_pattern, na=False)]
    print(f"  Absolute time format (HH:MM:SS.mmm or MM:SS.mmm): {len(absolute_times)} ({len(absolute_times)/len(non_null_times)*100:.1f}% if non-null > 0)")
    if len(absolute_times) > 0:
        print(f"  Sample absolute times:")
        for val in absolute_times.head(10):
            print(f"    {repr(val)}")
    print()
    
    # Summary by position
    print("Time format by position (leader vs others):")
    if 'Position' in race_results.columns:
        leader_times = race_results[race_results['Position'] == 1]['Time']
        other_times = race_results[race_results['Position'] != 1]['Time']
        
        leader_str = leader_times.astype(str)
        other_str = other_times.astype(str)
        
        leader_gaps = leader_str[leader_str.str.strip().str.startswith('+', na=False)]
        other_gaps = other_str[other_str.str.strip().str.startswith('+', na=False)]
        
        print(f"  Leader (position=1):")
        print(f"    Total: {len(leader_times)}, Non-null: {leader_times.notna().sum()}")
        print(f"    Gap times: {len(leader_gaps)}")
        if len(leader_times[leader_times.notna()]) > 0:
            print(f"    Sample: {leader_times[leader_times.notna()].iloc[0]}")
        
        print(f"  Others (position != 1):")
        print(f"    Total: {len(other_times)}, Non-null: {other_times.notna().sum()}")
        if other_times.notna().sum() > 0:
            print(f"    Gap times: {len(other_gaps)} ({len(other_gaps)/other_times.notna().sum()*100:.1f}%)")
        if len(other_times[other_times.notna()]) > 0:
            print(f"    Sample: {other_times[other_times.notna()].iloc[0]}")
else:
    print("⚠ 'Time' column not found in FastF1 results")



FASTF1 TIME COLUMN FORMAT ANALYSIS

Data type: object
Total values: 479
Non-null values: 424
Null values: 55

Sample Time values (first 20 non-null):
   1. '0 days 01:31:44.742000' (type: str)
   2. '0 days 00:00:22.457000' (type: str)
   3. '0 days 00:00:25.110000' (type: str)
   4. '0 days 00:00:39.669000' (type: str)
   5. '0 days 00:00:46.788000' (type: str)
   6. '0 days 00:00:48.458000' (type: str)
   7. '0 days 00:00:50.324000' (type: str)
   8. '0 days 00:00:56.082000' (type: str)
   9. '0 days 00:01:14.887000' (type: str)
  10. '0 days 00:01:33.216000' (type: str)
  11. '0 days 00:00:06.759000' (type: str)
  12. '0 days 00:00:08.316000' (type: str)
  13. '0 days 00:00:08.958000' (type: str)
  14. '0 days 00:00:09.482000' (type: str)
  15. '0 days 00:00:11.886000' (type: str)
  16. '0 days 00:00:17.632000' (type: str)
  17. '0 days 00:00:31.450000' (type: str)
  18. '0 days 00:00:32.417000' (type: str)
  19. '0 days 00:01:23.230000' (type: str)
  20. '0 days 00:00:20.795000' (

In [4]:
print("="*80)
print("FASTF1 OTHER KEY COLUMNS FORMAT ANALYSIS")
print("="*80)
print()

key_columns = ['GridPosition', 'Position', 'Points', 'Laps', 'Q1', 'Q2', 'Q3', 'Status']

for col in key_columns:
    if col in race_results.columns:
        print(f"{col}:")
        print(f"  Data type: {race_results[col].dtype}")
        print(f"  Non-null: {race_results[col].notna().sum()}/{len(race_results)}")
        
        # Show sample values
        non_null = race_results[col][race_results[col].notna()]
        if len(non_null) > 0:
            unique_count = non_null.nunique()
            print(f"  Unique values: {unique_count}")
            print(f"  Sample values:")
            for val in non_null.head(10):
                print(f"    {repr(val)}")
        print()


FASTF1 OTHER KEY COLUMNS FORMAT ANALYSIS

GridPosition:
  Data type: float64
  Non-null: 479/479
  Unique values: 21
  Sample values:
    1.0
    5.0
    4.0
    2.0
    3.0
    7.0
    9.0
    8.0
    6.0
    12.0

Position:
  Data type: float64
  Non-null: 479/479
  Unique values: 20
  Sample values:
    1.0
    2.0
    3.0
    4.0
    5.0
    6.0
    7.0
    8.0
    9.0
    10.0

Points:
  Data type: float64
  Non-null: 479/479
  Unique values: 19
  Sample values:
    26.0
    18.0
    15.0
    12.0
    10.0
    8.0
    6.0
    4.0
    2.0
    1.0

Laps:
  Data type: float64
  Non-null: 479/479
  Unique values: 43
  Sample values:
    57.0
    57.0
    57.0
    57.0
    57.0
    57.0
    57.0
    57.0
    57.0
    57.0

Q1:
  Data type: object
  Non-null: 0/479

Q2:
  Data type: object
  Non-null: 0/479

Q3:
  Data type: object
  Non-null: 0/479

Status:
  Data type: object
  Non-null: 479/479
  Unique values: 5
  Sample values:
    'Finished'
    'Finished'
    'Finished'
    'Fini

## 2. Master Data Format Inspection

Inspect master_races_clean.csv data types and formats for comparison.


In [5]:
print("="*80)
print("MASTER DATA FORMAT INSPECTION")
print("="*80)
print()

# Load master 2024 data
master = pd.read_csv(PROCESSED_DATA_DIR / 'master_races_clean.csv', low_memory=False)
master_2024 = master[master['year'] == 2024].copy()

print(f"Master 2024 rows: {len(master_2024):,}")
print(f"Master 2024 columns: {len(master_2024.columns)}")
print()

# Key columns to inspect
inspect_cols = ['time', 'milliseconds', 'grid', 'position', 'points', 'laps', 'q1', 'q2', 'q3', 'statusId']

for col in inspect_cols:
    if col in master_2024.columns:
        print(f"{col}:")
        print(f"  Data type: {master_2024[col].dtype}")
        print(f"  Non-null: {master_2024[col].notna().sum()}/{len(master_2024)}")
        
        # Show sample values
        non_null = master_2024[col][master_2024[col].notna()]
        if len(non_null) > 0:
            print(f"  Sample values:")
            for val in non_null.head(10):
                print(f"    {repr(val)} (type: {type(val).__name__})")
        print()


MASTER DATA FORMAT INSPECTION



Master 2024 rows: 479
Master 2024 columns: 61

time:
  Data type: object
  Non-null: 288/479
  Sample values:
    '1:31:44.742' (type: str)
    '1:32:07.199' (type: str)
    '1:32:09.852' (type: str)
    '1:32:24.411' (type: str)
    '1:32:31.530' (type: str)
    '1:32:33.200' (type: str)
    '1:32:35.066' (type: str)
    '1:32:40.824' (type: str)
    '1:32:59.629' (type: str)
    '1:33:17.958' (type: str)

milliseconds:
  Data type: float64
  Non-null: 288/479
  Sample values:
    5504742.0 (type: float)
    5527199.0 (type: float)
    5529852.0 (type: float)
    5544411.0 (type: float)
    5551530.0 (type: float)
    5553200.0 (type: float)
    5555066.0 (type: float)
    5560824.0 (type: float)
    5579629.0 (type: float)
    5597958.0 (type: float)

grid:
  Data type: int64
  Non-null: 479/479
  Sample values:
    1 (type: int)
    5 (type: int)
    4 (type: int)
    2 (type: int)
    3 (type: int)
    7 (type: int)
    9 (type: int)
    8 (type: int)
    6 (type: int)
    12 (type

## 2.5 FastF1 Data Format Inspection

In [13]:
print("="*80)
print("FASTF1 DATA FORMAT INSPECTION (Race + Qualifying Sessions)")
print("="*80)
print()

# Ensure FastF1 results were loaded
if 'fastf1_results' not in globals():
    raise NameError("fastf1_results not found. Ensure FastF1 results were loaded.")

race_results = fastf1_results[fastf1_results['Session'] == 'R'].copy()
quali_results = fastf1_results[fastf1_results['Session'] == 'Q'].copy()

print(f"FastF1 Race rows: {len(race_results):,}")
print(f"FastF1 Qualifying rows: {len(quali_results):,}")
print()

fastf1_col_map = {
    "time": "Time",
    "milliseconds": "Milliseconds",  # usually not present in FastF1 results
    "grid": "GridPosition",
    "position": "Position",
    "points": "Points",
    "laps": "Laps",
    "q1": "Q1",
    "q2": "Q2",
    "q3": "Q3",
}

inspect_cols = ["time", "milliseconds", "grid", "position", "points", "laps", "q1", "q2", "q3"]

for col in inspect_cols:
    fastf1_col = fastf1_col_map.get(col, col)

    # Q1/Q2/Q3 come from qualifying session; everything else from race session
    source_df = quali_results if col in ["q1", "q2", "q3"] else race_results

    if fastf1_col in source_df.columns:
        series = source_df[fastf1_col]
        print(f"{col} (FastF1.{fastf1_col}, Session={'Q' if col in ['q1','q2','q3'] else 'R'}):")
        print(f"  Data type: {series.dtype}")
        print(f"  Non-null: {series.notna().sum()}/{len(series)}")

        non_null = series[series.notna()]
        if len(non_null) > 0:
            print(f"  Sample values:")
            for val in non_null.head(10):
                print(f"    {repr(val)} (type: {type(val).__name__})")
        print()
    else:
        print(f"{col} (FastF1.{fastf1_col}, Session={'Q' if col in ['q1','q2','q3'] else 'R'}):")
        print(f"  ⚠ Column not found in FastF1 results")
        print()

FASTF1 DATA FORMAT INSPECTION (Race + Qualifying Sessions)

FastF1 Race rows: 479
FastF1 Qualifying rows: 479

time (FastF1.Time, Session=R):
  Data type: object
  Non-null: 424/479
  Sample values:
    '0 days 01:31:44.742000' (type: str)
    '0 days 00:00:22.457000' (type: str)
    '0 days 00:00:25.110000' (type: str)
    '0 days 00:00:39.669000' (type: str)
    '0 days 00:00:46.788000' (type: str)
    '0 days 00:00:48.458000' (type: str)
    '0 days 00:00:50.324000' (type: str)
    '0 days 00:00:56.082000' (type: str)
    '0 days 00:01:14.887000' (type: str)
    '0 days 00:01:33.216000' (type: str)

milliseconds (FastF1.Milliseconds, Session=R):
  ⚠ Column not found in FastF1 results

grid (FastF1.GridPosition, Session=R):
  Data type: float64
  Non-null: 479/479
  Sample values:
    1.0 (type: float)
    5.0 (type: float)
    4.0 (type: float)
    2.0 (type: float)
    3.0 (type: float)
    7.0 (type: float)
    9.0 (type: float)
    8.0 (type: float)
    6.0 (type: float)
    12.0

## 3. Format Comparison and Conversion Requirements

Compare FastF1 and master formats to identify conversion needs.


In [14]:
print("="*80)
print("FORMAT COMPARISON: FASTF1 vs MASTER")
print("="*80)
print()

# Compare Time/milliseconds
print("TIME/MILLISECONDS COMPARISON:")
print()

if 'Time' in race_results.columns and 'time' in master_2024.columns:
    fastf1_times = race_results['Time'][race_results['Time'].notna()]
    master_times = master_2024['time'][master_2024['time'].notna()]
    
    fastf1_str = fastf1_times.astype(str)
    master_str = master_times.astype(str)
    
    # Treat small timedeltas as gaps (FastF1 race gaps are stored as small timedeltas)
    fastf1_td = pd.to_timedelta(fastf1_times, errors="coerce")
    fastf1_gap_mask = fastf1_td.notna() & (fastf1_td < pd.Timedelta(minutes=10))
    fastf1_gaps = fastf1_times[fastf1_gap_mask]
    master_gaps = master_str[master_str.str.strip().str.startswith('+', na=False)]
    
    print(f"FastF1 Time:")
    print(f"  Total non-null: {len(fastf1_times)}")
    print(f"  Gap times (<10 min): {len(fastf1_gaps)} ({len(fastf1_gaps)/len(fastf1_times)*100:.1f}% if len > 0)")
    print(f"  Format: Timedelta strings (0 days 00:39:09.686000) or gap times (<10 min)")
    print()
    
    print(f"Master time:")
    print(f"  Total non-null: {len(master_times)}")
    print(f"  Gap times: {len(master_gaps)} ({len(master_gaps)/len(master_times)*100:.1f}% if len > 0)")
    print(f"  Format: String (HH:MM:SS.mmm or +MM:SS.mmm)")
    print()
    
    print(f"Master milliseconds:")
    if 'milliseconds' in master_2024.columns:
        master_ms = master_2024['milliseconds']
        master_ms_nonnull = master_ms[master_ms.notna()]
        master_ms_str = master_ms_nonnull.astype(str)
        
        backslash_n = master_ms_str[master_ms_str.str.strip().isin(['\\N', '\\\\N'])]
        master_ms_numeric = pd.to_numeric(master_ms_nonnull, errors='coerce')
        
        print(f"  Total non-null: {len(master_ms_nonnull)}")
        print(f"  \\\\N values: {len(backslash_n)}")
        print(f"  Numeric values: {master_ms_numeric.notna().sum()}")
        
        if master_ms_numeric.notna().sum() > 0:
            small_values = master_ms_numeric[(master_ms_numeric < 60000) & (master_ms_numeric > 0)]
            print(f"  Small values (< 60,000 ms): {len(small_values)} - LIKELY GAP TIMES!")
            if len(small_values) > 0:
                print(f"    ⚠ These need to be converted to absolute times")
                print(f"    Sample: {small_values.head(10).tolist()}")
    print()

# Compare Qualifying times
print("QUALIFYING TIMES COMPARISON:")
print()

if 'Session' in fastf1_results.columns:
    quali_results = fastf1_results[fastf1_results['Session'] == 'Q']
    
    for q_col in ['Q1', 'Q2', 'Q3']:
        if q_col in quali_results.columns and q_col.lower() in master_2024.columns:
            fastf1_q = quali_results[q_col][quali_results[q_col].notna()]
            master_q = master_2024[q_col.lower()][master_2024[q_col.lower()].notna()]
            
            fastf1_q_str = fastf1_q.astype(str)
            master_q_str = master_q.astype(str)
            
            fastf1_timedelta = fastf1_q_str[fastf1_q_str.str.contains('days', na=False)]
            master_mmss = master_q_str[master_q_str.str.match(r'^\d+:\d{2}\.\d+$', na=False)]
            
            print(f"{q_col}:")
            print(f"  FastF1: {len(fastf1_q)} non-null, {len(fastf1_timedelta)} timedelta format")
            print(f"  Master: {len(master_q)} non-null, {len(master_mmss)} MM:SS.mmm format")
            print(f"  → Conversion needed: Timedelta → MM:SS.mmm")
            print()


FORMAT COMPARISON: FASTF1 vs MASTER

TIME/MILLISECONDS COMPARISON:

FastF1 Time:
  Total non-null: 424
  Gap times (<10 min): 400 (94.3% if len > 0)
  Format: Timedelta strings (0 days 00:39:09.686000) or gap times (<10 min)

Master time:
  Total non-null: 288
  Gap times: 0 (0.0% if len > 0)
  Format: String (HH:MM:SS.mmm or +MM:SS.mmm)

Master milliseconds:
  Total non-null: 288
  \\N values: 0
  Numeric values: 288
  Small values (< 60,000 ms): 0 - LIKELY GAP TIMES!

QUALIFYING TIMES COMPARISON:

Q1:
  FastF1: 474 non-null, 474 timedelta format
  Master: 479 non-null, 479 MM:SS.mmm format
  → Conversion needed: Timedelta → MM:SS.mmm

Q2:
  FastF1: 358 non-null, 358 timedelta format
  Master: 371 non-null, 371 MM:SS.mmm format
  → Conversion needed: Timedelta → MM:SS.mmm

Q3:
  FastF1: 235 non-null, 235 timedelta format
  Master: 251 non-null, 251 MM:SS.mmm format
  → Conversion needed: Timedelta → MM:SS.mmm



## 4. Conversion Function Testing

Test conversion functions on sample data to verify they work correctly.


In [18]:
print("="*80)
print("FASTF1 GAP → ABSOLUTE + MS (single race)")
print("="*80)
print()

if 'race_results' not in globals():
    raise NameError("race_results not found. Ensure FastF1 results were loaded and filtered to Session == 'R'.")

def fmt_td(td):
    if pd.isna(td):
        return "NaT"
    total_ms = int(round(td.total_seconds() * 1000))
    secs, ms = divmod(total_ms, 1000)
    mins, secs = divmod(secs, 60)
    hrs, mins = divmod(mins, 60)
    if hrs > 0:
        return f"{hrs}:{mins:02d}:{secs:02d}.{ms:03d}"
    return f"{mins}:{secs:02d}.{ms:03d}"

# Pick first race in the data
first_event = race_results['Event'].iloc[0]
first_year = race_results['Year'].iloc[0]
race_data = race_results[(race_results['Event'] == first_event) & (race_results['Year'] == first_year)].copy()

print(f"Race: {first_year} {first_event} | Drivers: {len(race_data)}\n")

# Leader time
leader_row = race_data[pd.to_numeric(race_data['Position'], errors="coerce") == 1].head(1)
if leader_row.empty:
    print("Could not find leader (Position == 1)")
else:
    leader_time = leader_row.iloc[0]['Time']
    leader_td = pd.to_timedelta(leader_time, errors="coerce")
    leader_ms = int(round(leader_td.total_seconds() * 1000)) if pd.notna(leader_td) else None

    print(f"Leader: {leader_row.iloc[0].get('Abbreviation','?')}")
    print(f"  Time: {leader_time}")
    print(f"  Abs:  {fmt_td(leader_td)} | {leader_ms} ms\n")

    # Convert all non-null times; treat <10min as gap
    gap_cutoff = pd.Timedelta(minutes=10)

    for _, row in race_data[race_data['Time'].notna()].iterrows():
        time_val = row['Time']
        td = pd.to_timedelta(time_val, errors="coerce")
        if pd.isna(td) or leader_td is pd.NaT:
            continue

        is_gap = td < gap_cutoff and pd.to_numeric(row['Position'], errors="coerce") != 1
        gap_ms = int(round(td.total_seconds() * 1000)) if is_gap else None
        abs_td = (leader_td + td) if is_gap else td
        abs_ms = int(round(abs_td.total_seconds() * 1000)) if pd.notna(abs_td) else None

        if is_gap:
            print(f"Pos {row.get('Position','?')} | {row.get('Abbreviation','?')}")
            print(f"  Gap: {time_val} → {gap_ms} ms")
            print(f"  Abs: {fmt_td(abs_td)} → {abs_ms} ms\n")

FASTF1 GAP → ABSOLUTE + MS (single race)

Race: 2024 Bahrain Grand Prix | Drivers: 20

Leader: VER
  Time: 0 days 01:31:44.742000
  Abs:  1:31:44.742 | 5504742 ms

Pos 2.0 | PER
  Gap: 0 days 00:00:22.457000 → 22457 ms
  Abs: 1:32:07.199 → 5527199 ms

Pos 3.0 | SAI
  Gap: 0 days 00:00:25.110000 → 25110 ms
  Abs: 1:32:09.852 → 5529852 ms

Pos 4.0 | LEC
  Gap: 0 days 00:00:39.669000 → 39669 ms
  Abs: 1:32:24.411 → 5544411 ms

Pos 5.0 | RUS
  Gap: 0 days 00:00:46.788000 → 46788 ms
  Abs: 1:32:31.530 → 5551530 ms

Pos 6.0 | NOR
  Gap: 0 days 00:00:48.458000 → 48458 ms
  Abs: 1:32:33.200 → 5553200 ms

Pos 7.0 | HAM
  Gap: 0 days 00:00:50.324000 → 50324 ms
  Abs: 1:32:35.066 → 5555066 ms

Pos 8.0 | PIA
  Gap: 0 days 00:00:56.082000 → 56082 ms
  Abs: 1:32:40.824 → 5560824 ms

Pos 9.0 | ALO
  Gap: 0 days 00:01:14.887000 → 74887 ms
  Abs: 1:32:59.629 → 5579629 ms

Pos 10.0 | STR
  Gap: 0 days 00:01:33.216000 → 93216 ms
  Abs: 1:33:17.958 → 5597958 ms

Pos 11.0 | ZHO
  Gap: 0 days 00:00:06.75900

## 6. Standardized Format Recommendations

Based on analysis, provide recommendations for standardized formats.


In [11]:
print("="*80)
print("STANDARDIZED FORMAT RECOMMENDATIONS")
print("="*80)
print()

recommendations = {
    'time': {
        'format': 'String: HH:MM:SS.mmm or MM:SS.mmm (absolute race time only)',
        'conversion': 'Convert gap times to absolute before storing',
        'notes': 'Gap times should be converted by adding to leader time'
    },
    'milliseconds': {
        'format': 'Integer: milliseconds (absolute race time only)',
        'conversion': 'Convert from time string after gap conversion',
        'notes': 'Should be > 60,000 ms (1 minute) for valid race times. Smaller values are likely gaps.'
    },
    'q1': {
        'format': 'String: MM:SS.mmm',
        'conversion': 'Convert from FastF1 timedelta format (0 days 00:01:23.821000)',
        'notes': 'Extract MM:SS.mmm from timedelta string'
    },
    'q2': {
        'format': 'String: MM:SS.mmm',
        'conversion': 'Convert from FastF1 timedelta format',
        'notes': 'Same as q1'
    },
    'q3': {
        'format': 'String: MM:SS.mmm',
        'conversion': 'Convert from FastF1 timedelta format',
        'notes': 'Same as q1'
    },
    'position': {
        'format': 'Integer (nullable)',
        'conversion': 'Convert from float/string to Int64',
        'notes': 'Handle \\\\N and NaN values'
    },
    'grid': {
        'format': 'Integer (nullable)',
        'conversion': 'Convert to numeric, handle 0 values',
        'notes': '0 may indicate DNS/DQ'
    }
}

for col, rec in recommendations.items():
    print(f"{col}:")
    print(f"  Standard format: {rec['format']}")
    print(f"  Conversion needed: {rec['conversion']}")
    print(f"  Notes: {rec['notes']}")
    print()

print("="*80)
print("SUMMARY")
print("="*80)
print()
print("Key findings:")
print("1. FastF1 Time column: Mix of absolute times (timedelta) and gap times (+prefix)")
print("2. Gap times must be converted to absolute before calculating milliseconds")
print("3. Master milliseconds may contain gap times (small values < 60,000 ms)")
print("4. Qualifying times need format conversion: timedelta → MM:SS.mmm")
print("5. Position/grid need type conversion and \\\\N handling")
print()
print("Next steps:")
print("1. Implement gap time conversion in extract_race_data()")
print("2. Ensure all times are absolute before milliseconds conversion")
print("3. Normalize qualifying time formats")
print("4. Handle \\\\N values consistently")


STANDARDIZED FORMAT RECOMMENDATIONS

time:
  Standard format: String: HH:MM:SS.mmm or MM:SS.mmm (absolute race time only)
  Conversion needed: Convert gap times to absolute before storing
  Notes: Gap times should be converted by adding to leader time

milliseconds:
  Standard format: Integer: milliseconds (absolute race time only)
  Conversion needed: Convert from time string after gap conversion
  Notes: Should be > 60,000 ms (1 minute) for valid race times. Smaller values are likely gaps.

q1:
  Standard format: String: MM:SS.mmm
  Conversion needed: Convert from FastF1 timedelta format (0 days 00:01:23.821000)
  Notes: Extract MM:SS.mmm from timedelta string

q2:
  Standard format: String: MM:SS.mmm
  Conversion needed: Convert from FastF1 timedelta format
  Notes: Same as q1

q3:
  Standard format: String: MM:SS.mmm
  Conversion needed: Convert from FastF1 timedelta format
  Notes: Same as q1

position:
  Standard format: Integer (nullable)
  Conversion needed: Convert from float/